# Strange Places v5.0 - Interactive Demo

Explore **421,934 real-world mysterious phenomena** from 14 categories including caves, ancient megaliths, UFO sightings, meteorites, tornadoes, ghost towns, and more!

**Dataset Highlights:**
- 🌍 99.9% valid coordinates across all continents
- ⚖️ Perfectly balanced (no category > 20%)
- ✅ 100% real data from NASA, NOAA, USGS, OpenStreetMap
- 📊 14 categories, 421,934 georeferenced records

## Setup

Install required packages:

In [ ]:
# Uncomment to install dependencies
# !pip install pandas matplotlib seaborn folium plotly

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries loaded successfully")

## 1. Load Dataset

Load the complete Strange Places v5.0 dataset (164 MB JSON file).

In [ ]:
# Load dataset
print("Loading dataset...")
with open('all_phenomena_unified_v5_balanced.json', 'r') as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(data)

print(f"✓ Loaded {len(df):,} records")
print(f"✓ Columns: {', '.join(df.columns.tolist())}")

## 2. Dataset Overview

Explore the structure and basic statistics.

In [ ]:
# Display first few records
df.head(10)

In [ ]:
# Dataset info
print("Dataset Information:")
print(f"Total records: {len(df):,}")
print(f"Total categories: {df['category'].nunique()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nCoordinate coverage:")
print(f"  Valid lat/lon: {df[['latitude', 'longitude']].notna().all(axis=1).sum():,}")
print(f"  Missing coords: {df[['latitude', 'longitude']].isna().any(axis=1).sum():,}")

## 3. Category Distribution

Visualize the perfectly balanced distribution across all 14 categories.

In [ ]:
# Category counts
category_counts = df['category'].value_counts()
category_pct = (category_counts / len(df) * 100).round(1)

print("Category Distribution:")
print("=" * 70)
for cat, count in category_counts.items():
    pct = category_pct[cat]
    bar = '█' * int(pct)
    print(f"{cat:30s} {count:8,} ({pct:5.1f}%) {bar}")

print(f"\nBalance Rating: {'✅ EXCELLENT' if category_pct.max() < 20 else '⚠️ NEEDS IMPROVEMENT'}")
print(f"Largest category: {category_counts.index[0]} ({category_pct.iloc[0]:.1f}%)")

In [ ]:
# Visualize category distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
category_counts.plot(kind='barh', ax=ax1, color='steelblue')
ax1.set_xlabel('Number of Records', fontsize=12)
ax1.set_ylabel('Category', fontsize=12)
ax1.set_title('Records by Category', fontsize=14, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# Pie chart
category_counts.plot(kind='pie', ax=ax2, autopct='%1.1f%%', startangle=90)
ax2.set_ylabel('')
ax2.set_title('Category Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Geographic Distribution

Explore the global distribution of phenomena.

In [ ]:
# Geographic bounds
print("Geographic Coverage:")
print(f"Latitude range: {df['latitude'].min():.2f}° to {df['latitude'].max():.2f}°")
print(f"Longitude range: {df['longitude'].min():.2f}° to {df['longitude'].max():.2f}°")

# Hemisphere distribution
print("\nHemisphere Distribution:")
print(f"Northern Hemisphere: {(df['latitude'] >= 0).sum():,} ({(df['latitude'] >= 0).sum() / len(df) * 100:.1f}%)")
print(f"Southern Hemisphere: {(df['latitude'] < 0).sum():,} ({(df['latitude'] < 0).sum() / len(df) * 100:.1f}%)")
print(f"Eastern Hemisphere: {(df['longitude'] >= 0).sum():,} ({(df['longitude'] >= 0).sum() / len(df) * 100:.1f}%)")
print(f"Western Hemisphere: {(df['longitude'] < 0).sum():,} ({(df['longitude'] < 0).sum() / len(df) * 100:.1f}%)")

In [ ]:
# Scatter plot of all phenomena
fig, ax = plt.subplots(figsize=(16, 8))

# Sample 10,000 points for performance
sample = df.sample(min(10000, len(df)))

# Plot by category
for category in sample['category'].unique():
    cat_data = sample[sample['category'] == category]
    ax.scatter(cat_data['longitude'], cat_data['latitude'], 
               alpha=0.5, s=1, label=category)

ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Global Distribution of Phenomena (10K sample)', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Interactive Map Visualization

Create an interactive Folium map showing phenomena locations.

In [ ]:
import folium
from folium.plugins import HeatMap, MarkerCluster

# Create base map
m = folium.Map(location=[20, 0], zoom_start=2, tiles='CartoDB positron')

# Sample data for performance (use first 1000 records)
sample = df.sample(min(1000, len(df)))

# Color map for categories
color_map = {
    'osm_caves': 'brown',
    'megalithic_portal': 'darkpurple',
    'ufo_sightings': 'red',
    'nasa_meteorites': 'orange',
    'noaa_tornadoes': 'blue',
    'osm_ghost_towns': 'gray',
    'usgs_waterfalls': 'lightblue',
    'noaa_storm_events': 'darkblue',
    'usgs_earthquakes': 'darkred',
    'nasa_fireballs': 'yellow',
    'usgs_volcanoes': 'red',
    'noaa_thermal_springs': 'pink',
    'noaa_shipwrecks': 'darkgreen'
}

# Add markers
marker_cluster = MarkerCluster().add_to(m)

for idx, row in sample.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        popup=f"{row.get('name', 'Unknown')}<br>{row['category']}",
        color=color_map.get(row['category'], 'gray'),
        fill=True,
        fillOpacity=0.6
    ).add_to(marker_cluster)

# Save and display
m.save('phenomena_map.html')
print("✓ Interactive map saved to phenomena_map.html")
m

## 6. Category-Specific Analysis

Dive deeper into specific categories.

In [ ]:
# Analyze UFO sightings by decade
ufo_df = df[df['category'] == 'ufo_sightings'].copy()
ufo_df['year'] = pd.to_datetime(ufo_df['date'], errors='coerce').dt.year
ufo_df['decade'] = (ufo_df['year'] // 10 * 10).astype('Int64')

ufo_by_decade = ufo_df['decade'].value_counts().sort_index()

plt.figure(figsize=(14, 6))
ufo_by_decade.plot(kind='bar', color='crimson')
plt.xlabel('Decade', fontsize=12)
plt.ylabel('Number of Sightings', fontsize=12)
plt.title('UFO Sightings by Decade', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Total UFO sightings: {len(ufo_df):,}")
print(f"Date range: {ufo_df['year'].min():.0f} - {ufo_df['year'].max():.0f}")
print(f"Peak decade: {ufo_by_decade.idxmax():.0f}s ({ufo_by_decade.max():,} sightings)")

In [ ]:
# Analyze tornado magnitudes
tornado_df = df[df['category'] == 'noaa_tornadoes'].copy()

if 'magnitude' in tornado_df.columns:
    magnitude_counts = tornado_df['magnitude'].value_counts().sort_index()
    
    plt.figure(figsize=(12, 6))
    magnitude_counts.plot(kind='bar', color='steelblue')
    plt.xlabel('F-Scale/EF-Scale', fontsize=12)
    plt.ylabel('Number of Tornadoes', fontsize=12)
    plt.title('Tornado Distribution by Magnitude (F-Scale/EF-Scale)', fontsize=14, fontweight='bold')
    plt.xticks(rotation=0)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"Total tornadoes: {len(tornado_df):,}")
    print(f"Most common magnitude: F{magnitude_counts.idxmax():.0f} ({magnitude_counts.max():,} tornadoes)")

## 7. Regional Analysis

Filter and analyze phenomena by geographic region.

In [ ]:
# Define regions
def categorize_region(lat, lon):
    if 24 <= lat <= 49 and -125 <= lon <= -66:
        return 'USA (Continental)'
    elif 36 <= lat <= 71 and -10 <= lon <= 40:
        return 'Europe'
    elif -10 <= lat <= 37 and -20 <= lon <= 51:
        return 'Africa'
    elif 10 <= lat <= 54 and 60 <= lon <= 150:
        return 'Asia (East)'
    elif -55 <= lat <= -10 and 110 <= lon <= 155:
        return 'Australia'
    elif -56 <= lat <= 15 and -82 <= lon <= -34:
        return 'South America'
    else:
        return 'Other'

df['region'] = df.apply(lambda x: categorize_region(x['latitude'], x['longitude']), axis=1)

# Regional distribution
region_counts = df['region'].value_counts()
print("Phenomena by Region:")
print("=" * 50)
for region, count in region_counts.items():
    pct = count / len(df) * 100
    print(f"{region:20s} {count:8,} ({pct:5.1f}%)")

# Visualize
plt.figure(figsize=(12, 6))
region_counts.plot(kind='barh', color='teal')
plt.xlabel('Number of Records', fontsize=12)
plt.ylabel('Region', fontsize=12)
plt.title('Phenomena by Geographic Region', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Query Examples

Practical examples of filtering and querying the dataset.

In [ ]:
# Example 1: Find all caves in France
france_caves = df[
    (df['category'] == 'osm_caves') & 
    (df['latitude'].between(42, 51)) &
    (df['longitude'].between(-5, 10))
]
print(f"Caves in France: {len(france_caves):,}")
print(france_caves[['name', 'latitude', 'longitude']].head())

In [ ]:
# Example 2: Find megalithic sites within 100km of Stonehenge
from math import radians, sin, cos, sqrt, asin

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance in km between two points"""
    R = 6371  # Earth radius in km
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return 2 * R * asin(sqrt(a))

# Stonehenge coordinates
stonehenge_lat, stonehenge_lon = 51.1789, -1.8262

megaliths = df[df['category'] == 'megalithic_portal'].copy()
megaliths['distance'] = megaliths.apply(
    lambda x: haversine_distance(stonehenge_lat, stonehenge_lon, x['latitude'], x['longitude']),
    axis=1
)

nearby_megaliths = megaliths[megaliths['distance'] <= 100].sort_values('distance')
print(f"Megalithic sites within 100km of Stonehenge: {len(nearby_megaliths)}")
print(nearby_megaliths[['name', 'distance', 'latitude', 'longitude']].head(10))

In [ ]:
# Example 3: Find the largest meteorites
meteorites = df[df['category'] == 'nasa_meteorites'].copy()

if 'mass' in meteorites.columns:
    largest_meteorites = meteorites.nlargest(10, 'mass')
    print("Top 10 Largest Meteorites:")
    print(largest_meteorites[['name', 'mass', 'latitude', 'longitude']].to_string())
else:
    print("Mass data not available in this category")

## 9. Statistical Summary

Generate comprehensive statistics about the dataset.

In [ ]:
print("STRANGE PLACES v5.0 - STATISTICAL SUMMARY")
print("=" * 70)
print(f"\nTotal Records: {len(df):,}")
print(f"Total Categories: {df['category'].nunique()}")
print(f"\nGeographic Coverage:")
print(f"  Valid coordinates: {df[['latitude', 'longitude']].notna().all(axis=1).sum():,} ({df[['latitude', 'longitude']].notna().all(axis=1).sum() / len(df) * 100:.1f}%)")
print(f"  Latitude range: {df['latitude'].min():.2f}° to {df['latitude'].max():.2f}°")
print(f"  Longitude range: {df['longitude'].min():.2f}° to {df['longitude'].max():.2f}°")
print(f"\nCategory Balance:")
print(f"  Largest category: {category_counts.index[0]} ({category_pct.iloc[0]:.1f}%)")
print(f"  Smallest category: {category_counts.index[-1]} ({category_pct.iloc[-1]:.1f}%)")
print(f"  Balance rating: {'✅ EXCELLENT' if category_pct.max() < 20 else '⚠️ NEEDS IMPROVEMENT'}")
print(f"\nData Quality:")
print(f"  Records with names: {df['name'].notna().sum():,} ({df['name'].notna().sum() / len(df) * 100:.1f}%)")
print(f"  Records with dates: {df['date'].notna().sum():,} ({df['date'].notna().sum() / len(df) * 100:.1f}%)")
print(f"  Records with descriptions: {df['description'].notna().sum():,} ({df['description'].notna().sum() / len(df) * 100:.1f}%)")
print("\n" + "=" * 70)

## 10. Export Filtered Subsets

Create filtered datasets for specific use cases.

In [ ]:
# Export USA-only phenomena
usa_data = df[
    (df['latitude'].between(24, 49)) &
    (df['longitude'].between(-125, -66))
]

usa_data.to_json('strange_places_usa_only.json', orient='records', indent=2)
print(f"✓ Exported {len(usa_data):,} USA records to strange_places_usa_only.json")

# Export natural phenomena only
natural_categories = ['osm_caves', 'usgs_waterfalls', 'noaa_thermal_springs', 
                      'usgs_volcanoes', 'usgs_earthquakes']
natural_data = df[df['category'].isin(natural_categories)]

natural_data.to_json('strange_places_natural_only.json', orient='records', indent=2)
print(f"✓ Exported {len(natural_data):,} natural phenomena to strange_places_natural_only.json")

# Export ancient/archaeological only
ancient_categories = ['megalithic_portal', 'osm_ghost_towns']
ancient_data = df[df['category'].isin(ancient_categories)]

ancient_data.to_json('strange_places_ancient_only.json', orient='records', indent=2)
print(f"✓ Exported {len(ancient_data):,} ancient sites to strange_places_ancient_only.json")

## Conclusion

This demo notebook showcased:

✅ **Loading and exploring** 421K+ real-world phenomena  
✅ **Visualizing** category distribution and balance  
✅ **Mapping** global geographic coverage  
✅ **Analyzing** specific categories (UFOs, tornadoes, meteorites)  
✅ **Querying** by region, distance, and attributes  
✅ **Exporting** filtered subsets  

### Next Steps

- **Machine Learning**: Train clustering models, anomaly detection
- **Deep Dive**: Analyze temporal patterns, correlations between categories
- **Advanced Mapping**: Create interactive dashboards with Plotly/Dash
- **Research**: Publish findings, contribute to earth science

### Resources

- **Full Documentation**: README_V5.md
- **Release Notes**: V5_RELEASE_NOTES.md
- **Quick Start**: QUICK_START_V5.md
- **GitHub**: https://github.com/lukesteuber/strange-places-dataset

**Author**: Luke Steuber | luke@lukesteuber.com | @lukesteuber.com (Bluesky)